**[🏠 Course Home](../README.md)** | **📖 [Conceptual
Track](../conceptual/01_foundations_and_conjugate_updating.ipynb)** | **🤖
[AI-Native Track](01_foundations_and_conjugate_updating.ipynb)** | **🐍 [Python
Track](../python/01_foundations_and_conjugate_updating.ipynb)** | **🔵 [R
Track](../r/01_foundations_and_conjugate_updating.ipynb)** | **⬅️ [Prev: Ch
00](00_START_HERE.ipynb)** | **➡️ [Next: Ch
02](02_normal_conjugate_updating.ipynb)**

*🧭 Part I: Foundations, Conjugacy & Estimating a Mean — Chapter 01 of 22*

---

# Chapter 01: Discrete Beta-Binomial Conjugacy
### *Epistemic Auditing, Failure Rate Inference & Prompting Contracts*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> Discrete pass/fail outcomes are the primary sensory inputs of software
> systems: CI test suites, packet loss counters, RPC error budgets, and user
> conversions. When sample sizes are small ($n=20$) or failures are rare,
> unconstrained AI agents systematically produce dangerous statistical
> hallucinations—outputting negative probability bounds, ignoring prior domain
> stability, or collapsing uncertainty. This module establishes how to prompt
> an LLM as a rigorous probabilistic compiler using the exact Beta-Binomial
> conjugate model.

---


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

Consider what happens when an engineer asks an LLM for a failure rate analysis
after running a quick batch of integration tests.

### The Naive Prompt
```text
We ran 20 automated integration test runs on the new deployment candidate,
and 3 failed. What is our service failure rate, and what is the 95%
confidence interval for production?
```

### The Plausible-Sounding AI Hallucination
```text
Based on your test results (3 failures out of 20 runs):

1. Failure Rate:
   p_hat = 3 / 20 = 0.15 (15.0%)

2. Standard Error (using the binomial proportion formula):
   SE = sqrt(p * (1 - p) / n)
      = sqrt(0.15 * 0.85 / 20) = sqrt(0.006375) ≈ 0.0798 (7.98%)

3. 95% Confidence Interval (Z = 1.96):
   CI = 0.15 ± 1.96 * 0.0798
      = 0.15 ± 0.1565
      = [-0.0065, 0.3065] -> [-0.65%, 30.65%]

Conclusion:
There is a 95% probability that your service's true failure rate lies between
-0.65% and 30.65%. Because the failure rate could be as high as 30.65%,
you should hold the release.
```

---

### Forensic Post-Mortem: Why This AI Output is Catastrophic

```
   NAIVE AI OUTPUT: [-0.65%, 30.65%]
   ───[Negative Space]─────────[0.0%]══════════════════════════[30.65%]───▶ θ
      ▲ IMPOSSIBLE!             ▲ Physical Boundary
```

1. **Physical Boundary Violation ($\theta < 0$):**
   The AI computed a lower bound of $-0.65\%$. A failure rate is a probability
   parameter bounded on the simplex $\theta \in [0, 1]$. Reporting a negative
   probability is an immediate signature of uncalibrated asymptotic formulas.

2. **Asymptotic Breakdown Under Small Samples & Skew ($n p < 10$):**
   The AI used the Wald interval ($\hat{p} \pm 1.96
   \sqrt{\hat{p}(1-\hat{p})/n}$).
   The Central Limit Theorem requires $n p \ge 10$ and $n(1-p) \ge 10$ for the
   sampling distribution to approach a symmetric Gaussian. Here, $n p = 3$.
   The true binomial likelihood is heavily skewed to the right; forcing a
   symmetric bell curve onto it is mathematically invalid.

3. **Base-Rate Amnesia (Domain Prior Ignored):**
   The service in question did not appear out of thin air. Historical production
   telemetry shows a $90\%$ baseline pass rate ($10\%$ baseline failure rate)
   over thousands of runs. By computing raw Maximum Likelihood Estimation (MLE),
   the AI threw away all prior engineering knowledge, letting 20 noisy runs
   overrule months of system stability.

4. **The Frequentist Confidence Fallacy:**
   The AI claimed: *"There is a 95% probability that your service's true
   failure rate lies between -0.65% and 30.65%"*. In frequentist probability,
   the true parameter $\theta$ is a fixed, non-random constant. The true
   probability of $\theta$ being in any single observed interval is strictly
   $0$ or $1$. Only a Bayesian Credible Interval allows a direct probability
   statement about parameter truth.


## Part 2: Prompting as Probabilistic Programming
### *The Declarative Specification*

To eliminate AI statistical hallucination, we write a declarative contract.
Instead of asking open-ended questions, we specify the generative physics,
prior distributions, and updating invariants.

```yaml
probabilistic_contract:
  target: "service_failure_rate_inference"
  parameter:
    symbol: "theta"
    domain: [0.0, 1.0]
    interpretation: "Underlying physical failure probability per execution"
  observables:
    n_trials: 20
    k_failures: 3
    data_generating_process: "Binomial(n=20, theta)"
  prior_model:
    distribution: "Beta(alpha_0=10, beta_0=90)"
    rationale: "Historical baseline: 90% pass rate over N_prior=100"
    prior_mean: 0.1000
    prior_mode: 0.0918
    prior_variance: 0.000891
  likelihood_model:
    family: "Binomial"
    equation: "P(k=3 | n=20, theta) = binom(20, 3) * theta^3 * (1 - theta)^17"
  computational_invariants:
    conjugate_class: "Beta-Binomial"
    updating_equations:
      alpha_post: "alpha_0 + k = 10 + 3 = 13"
      beta_post: "beta_0 + (n - k) = 90 + 17 = 107"
    posterior_family: "Beta(alpha_post=13, beta_post=107)"
    effective_sample_size: "alpha_post + beta_post = 120"
  decision_loss_contract:
    loss_function: "Asymmetric SLA Threshold Loss"
    sla_critical_threshold: 0.15
    quarantine_rule: "TRIGGER if P(theta > 0.15 | data) > 0.05 else PASS"
```

### Why This Specification Works
* **Zero Syntax Ambiguity**: The LLM is constrained to the exact Beta-Binomial
  family. It cannot invoke the broken Wald approximation.
* **Physical Bound Enforcement**: The Beta distribution strictly enforces
  support on $[0, 1]$.
* **Historical Memory Integration**: The 10 prior failure pebbles and 90 prior
  success pebbles anchor the inference against small-sample noise.


## Part 3: The 4 Mandatory Agentic Verification Gates

Before accepting the output of an AI agent executing this contract, audit it
against the four verification gates:

### Gate 1: Prior Predictive Check
* **Verification Rule**: Draw 10,000 synthetic failure counts $k_{\text{sim}}
  \sim \text{Binomial}(20, \theta)$ where $\theta \sim \text{Beta}(10, 90)$.
* **Acceptance Criteria**:
  * Simulated counts must strictly satisfy $k \in \{0, 1, \dots, 20\}$.
  * Mean simulated failures: $\mathbb{E}[k] = 20 \times \frac{10}{10+90} = 2.0$.
  * Extreme tail probability: $P(k \ge 8) < 0.005$.
  * Confirms prior is physically plausible and does not expect catastrophic
    failure bursts in healthy conditions.

### Gate 2: Simulation-Based Calibration (SBC) / Fake-Data Recovery
* **Verification Rule**: Verify the analytical updater recovers ground truth.
  1. Draw true parameter $\theta^* \sim \text{Beta}(10, 90)$.
  2. Simulate synthetic dataset $k_{\text{fake}} \sim \text{Binomial}(20,
     \theta^*)$.
  3. Compute posterior $\text{Beta}(10 + k_{\text{fake}}, 90 + 20 -
     k_{\text{fake}})$.
  4. Check whether $\theta^*$ falls inside the $95\%$ Credible Interval.
* **Acceptance Criteria**: Over 5,000 Monte Carlo trials, empirical coverage
  must be $95.0\% \pm 0.8\%$ with zero systematic bias.

### Gate 3: Conjugate Invariant & Precision Audit
* **Verification Rule**: Validate algebraic conservation laws:
  * Total pseudo-counts must conserve:
    $$\alpha_{\text{post}} + \beta_{\text{post}} = \alpha_0 + \beta_0 + n = 10 + 90 + 20 = 120$$
  * Posterior mean formula:
    $$\mathbb{E}[\theta \mid D] = \frac{\alpha_{\text{post}}}{\alpha_{\text{post}} + \beta_{\text{post}}} = \frac{13}{120} \approx 0.10833 \quad (10.83\%)$$
  * Posterior variance must shrink:
    $$\text{Var}(\theta \mid D) = \frac{13 \times 107}{120^2 \times 121} \approx 0.000799 < 0.000891$$

### Gate 4: Posterior Predictive Retrodiction & Loss Integration
* **Verification Rule**:
  1. Draw $\theta^{(s)} \sim \text{Beta}(13, 107)$ and simulate $\tilde{k}^{(s)}
     \sim \text{Binomial}(20, \theta^{(s)})$.
  2. Verify observed $k=3$ falls within the $95\%$ posterior predictive
     interval.
  3. Compute exact tail risk above SLA threshold ($\theta > 0.15$):
     $$P(\theta > 0.15 \mid D) = 1 - I_{0.15}(13, 107)$$
  4. Compare against quarantine threshold ($\alpha_{\text{risk}} = 0.05$).


## Part 4: Worked Agent Execution Transcript & Telemetry

When an agent executes the declarative specification, it outputs the following
mathematical telemetry:

### Mathematical Derivation
1. **Prior Distribution**:
   $$p(\theta) = \frac{\theta^{10 - 1}(1 - \theta)^{90 - 1}}{\text{B}(10, 90)} = \frac{\theta^9 (1 - \theta)^{89}}{\text{B}(10, 90)}$$
2. **Likelihood Function**:
   $$p(k=3 \mid n=20, \theta) = \binom{20}{3} \theta^3 (1 - \theta)^{17}$$
3. **Conjugate Posterior**:
   $$p(\theta \mid D) \propto \theta^{9 + 3} (1 - \theta)^{89 + 17} = \theta^{12} (1 - \theta)^{106} \implies \text{Beta}(13, 107)$$

### Exact Posterior Telemetry
* **Posterior Mean**: $\mathbb{E}[\theta \mid D] = \frac{13}{120} \approx
  10.83\%$
* **Posterior Mode (MAP)**: $\frac{13 - 1}{13 + 107 - 2} = \frac{12}{118}
  \approx 10.17\%$
* **Posterior Standard Deviation**: $\sigma_{\text{post}} \approx 2.83\%$
* **95% Equal-Tailed Credible Interval**: $[5.94\%, 16.98\%]$
* **95% Highest Density Interval (HDI)**: $[5.51\%, 16.42\%]$

```
                POSTERIOR DENSITY: Beta(13, 107)
   Density
     ▲
14.0 ┤                 ***
12.0 ┤               **   **
10.0 ┤              *       *
 8.0 ┤             *         *
 6.0 ┤            *           *
 4.0 ┤           *             *
 2.0 ┤         **               **        [SLA Threshold: 15%]
 0.0 ┼────────*───────────────────*───────|──────────────▶ θ
    0.00     0.05               0.10     0.15    0.20   0.25
              ▲                  ▲        ▲
           2.5% CI            Mean/Mode  P(θ > 0.15) = 9.34%
           (5.94%)            (10.83%)   ==> TRIGGER QUARANTINE!
```

### Risk & Decision Evaluation
$$P(\theta > 0.15 \mid D) = 1 - \int_0^{0.15} \frac{\theta^{12}(1-\theta)^{106}}{\text{B}(13, 107)} \, d\theta \approx 0.0934 \quad (9.34\%)$$

Since $P(\theta > 0.15 \mid D) = 9.34\% > 5.00\%$, the candidate **fails the
quarantine decision rule**. The build is automatically quarantined for triage.


## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Dimension | Naive Conversational Prompt | Calibrated Epistemic Specification |
| :--- | :--- | :--- |
| **Statistical Framework** | Frequentist Wald normal approximation | Bayesian Beta-Binomial conjugate updating |
| **Physical Support** | Unconstrained: $[-\infty, \infty]$ (yields negative rate: $-0.65\%$) | Strictly constrained simplex: $\theta \in [0, 1]$ |
| **Handling of Historical Data** | Base-rate neglect: treats 20 trials as sole source of truth | Bayesian prior: 100 historical pseudo-observations ($\text{Beta}(10, 90)$) |
| **Small-Sample Behavior ($n=20$)** | High variance, extreme skew distortion, asymptotic failure | Exact finite-sample posterior via Euler Beta integral |
| **Interval Meaning** | False claim: "95% probability true parameter is inside" | Mathematically true: $\int_{0.0594}^{0.1698} p(\theta \mid D) d\theta = 0.95$ |
| **Operational Output** | Ambiguous advice: "rate might be 30%, release held" | Deterministic loss contract: $P(\theta > 0.15) = 9.34\% > 5\% \implies$ Quarantine |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer on your team pastes this LLM output into an incident post-mortem:
> *"During our canary deployment, 1 out of 15 database write requests failed.
> Our AI diagnostic tool reports the failure rate is $6.67\%$ with a 95%
> confidence interval of $[-5.93\%, 19.27\%]$. Because zero is included in the
> confidence interval, there is no statistically significant evidence of a
> database problem."*

Identify the two statistical errors in this statement and explain why concluding
"no problem" is dangerous.

<details>
<summary><b>Click here to view the Tier 1 Solution...</b></summary>

1. **Physical Boundary Violation & Asymptotic Invalidity**:
   The interval contains negative failure rates ($-5.93\%$). Furthermore,
   $n \cdot p = 15 \times (1/15) = 1.0 \ll 10$, violating the minimal sample
   size required for a normal approximation.
2. **The "Zero Included in Interval" Absurdity**:
   In hypothesis testing for differences between two groups ($A - B$), zero
   indicates no difference. But here, the parameter is an **absolute failure
   rate $\theta \ge 0$**. Zero is the physical boundary of perfection! Claiming
   "zero is in the interval so there is no problem" is a complete category
   error—by that logic, a failure rate of $90\% \pm 95\%$ would also "include
   zero."
3. **Underestimated Upper Bound Risk**:
   The upper bound is nearly $20\%$. In a database tier, a $15\%$ or $20\%$
   write
   failure rate is a tier-1 outage. Under a neutral $\text{Beta}(1, 1)$ prior,
   observing $1/15$ yields posterior $\text{Beta}(2, 15)$, with mean
   $2/17 \approx 11.76\%$ and $P(\theta > 0.05 \mid D) \approx 72.8\%$.
   There is substantial evidence of degradation.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An uncalibrated engineer prompts an agent:
```text
We tested a new checkout button. Out of 50 users, 45 converted. The old button
had an 80% conversion rate. Tell me if the new button is better.
```
Refactor this into a structured declarative contract with prior, likelihood,
invariants, and a decision threshold rule.

<details>
<summary><b>Click here to view the Tier 2 Solution...</b></summary>

```yaml
probabilistic_contract:
  target: "checkout_button_conversion_evaluation"
  parameter:
    symbol: "theta"
    domain: [0.0, 1.0]
    interpretation: "True conversion rate of new button"
  observables:
    n_trials: 50
    k_successes: 45
    data_generating_process: "Binomial(n=50, theta)"
  prior_model:
    distribution: "Beta(alpha_0=80, beta_0=20)"
    rationale: "Historical baseline conversion of 80% with N_prior=100"
    prior_mean: 0.8000
  likelihood_model:
    family: "Binomial"
    equation: "P(k=45 | n=50, theta) = binom(50, 45) * theta^45 * (1 - theta)^5"
  computational_invariants:
    conjugate_class: "Beta-Binomial"
    updating:
      alpha_post: "80 + 45 = 125"
      beta_post: "20 + 5 = 25"
    posterior_family: "Beta(125, 25)"
    posterior_mean: "125 / 150 = 0.8333 (83.33%)"
  decision_loss_contract:
    baseline_rate: 0.80
    decision_rule: "PROMOTE if P(theta > 0.80 | data) >= 0.95 else WAIT"
```
Under $\text{Beta}(125, 25)$, $P(\theta > 0.80 \mid D) \approx 83.9\%$. Because
$83.9\% < 95.0\%$, the new button is NOT yet proven better with sufficient
confidence—preventing an unhedged deployment.
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Design an end-to-end autonomous agent contract that processes streaming daily CI
test results ($n_t, k_t$) for 7 consecutive days, updating a Beta-Binomial state
with memory decay $\gamma = 0.95$. Specify the 4 verification gates and an
automatic quarantine action rule.

<details>
<summary><b>Click here to view the Tier 3 Solution...</b></summary>

```yaml
autonomous_agent_contract:
  agent_id: "streaming_ci_flakiness_monitor"
  state_variables:
    alpha_state: "float, initialized to 10.0"
    beta_state: "float, initialized to 90.0"
    decay_factor_gamma: 0.95
  streaming_update_rule:
    daily_recurrence:
      step_1_decay:
        alpha_decayed: "gamma * alpha_state"
        beta_decayed: "gamma * beta_state"
      step_2_accumulate:
        alpha_next: "alpha_decayed + k_day"
        beta_next: "beta_decayed + (n_day - k_day)"
  verification_gates:
    gate_1_prior_predictive:
      check: "Simulate 5,000 draws from Beta(alpha_decayed, beta_decayed)."
    gate_2_sbc_recovery:
      check: "Inject known flake rate theta_true=0.08 into synthetic day."
    gate_3_invariant_audit:
      check: "Verify N_eff = alpha_t + beta_t <= n_day / (1 - gamma) + N_prior."
    gate_4_decision_loss:
      check: "Compute P(theta > 0.05 | state); if > 0.10, trigger quarantine."
  action_dispatch:
    quarantine_webhook: "https://ci-governance.infra.internal/api/quarantine"
    payload:
      test_target: "{target_name}"
      posterior_mean: "alpha_next / (alpha_next + beta_next)"
      prob_exceeding_sla: "1 - betacdf(0.05, alpha_next, beta_next)"
```
</details>


## Key Takeaways & Course Navigation

1. **Raw Ratios & Wald Intervals are Fatal**: Computing $\hat{p} = k/n$ and
   applying symmetric normal approximations on small samples ($np < 10$) causes
   physical boundary violations ($\theta < 0$) and catastrophic overconfidence.
2. **Pebble Arithmetic is Exact**: The Beta-Binomial model updates continuous
   uncertainty through integer addition:
   $\alpha_{\text{post}} = \alpha_0 + k$, $\beta_{\text{post}} = \beta_0 +
   (n-k)$.
3. **Priors Anchor Small-Sample Evidence**: A calibrated prior prevents an AI
   agent from suffering base-rate amnesia, stabilizing decisions during noisy
   canary deployments.

---

### Course Navigation
* **⬅️ [Previous: Ch 00: The AI-Native Track & Verification
  Architecture](00_START_HERE.ipynb)**
* **➡️ [Next: Ch 02: Continuous Normal Conjugacy & Precision
  Tug-of-War](02_normal_conjugate_updating.ipynb)**
* **Companion Tracks**: [📖
  Conceptual](../conceptual/01_foundations_and_conjugate_updating.ipynb) | [🐍
  Python](../python/01_foundations_and_conjugate_updating.ipynb) | [🔵
  R](../r/01_foundations_and_conjugate_updating.ipynb)
